# CV analyser: run the local-model eval on a Kaggle GPU

Runs the **same** evaluation as `scripts/run_eval.py` on a laptop, with the same
model (`qwen2.5:7b` through Ollama) and the same code from GitHub, but on Kaggle's
free GPU, so it takes minutes instead of over an hour.

**Before running:**
1. *Settings → Accelerator*: **GPU T4 x2** (or P100).
2. *Settings → Internet*: **On** (needs a phone-verified Kaggle account).
3. *Add Input*: your private dataset made from `cv-eval-data.zip`
   (fictional eval CVs, answer key, job ads). Never upload real people's CVs.
4. Run it in the background with **Save Version → Save & Run All (Commit)**, so it
   keeps going even if the browser tab sleeps. Interactive sessions stop when idle.
5. Results and logs are in the version's **Output** tab: `kaggle_qwen7b.json`,
   `eval.log` and `ollama.log`.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install and start Ollama

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

In [ ]:
import os
import subprocess
import time
import urllib.request

env = dict(os.environ, OLLAMA_KEEP_ALIVE="2h")
# The server runs for the whole session, so its log file stays open on purpose.
log = open("/kaggle/working/ollama.log", "w")  # noqa: SIM115
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /kaggle/working/ollama.log")
print("Ollama is running")
!ollama --version

In [ ]:
!ollama pull qwen2.5:7b 2>&1 | tail -n 2

## 3. Smoke test: one short answer, and is it on the GPU?

Fails fast here, with a clear message, instead of an hour into the eval.

In [ ]:
import json

payload = {"model": "qwen2.5:7b", "prompt": "Reply with one word: ready", "stream": False}
request = urllib.request.Request(
    "http://localhost:11434/api/generate",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)
answer = json.loads(urllib.request.urlopen(request, timeout=600).read())
print("Model says:", answer["response"].strip())
print(f"Speed: {answer['eval_count'] / (answer['eval_duration'] / 1e9):.1f} tokens/s (laptop CPU: ~4)")
!ollama ps

## 4. Get the code

In [ ]:
!rm -rf /kaggle/working/cv-analyser-rag
!git clone -q --depth 1 https://github.com/shahriar62b/cv-analyser-rag.git /kaggle/working/cv-analyser-rag
%cd /kaggle/working/cv-analyser-rag
!git log --oneline -1
# Install the project without the laptop's exact pins, so Kaggle keeps its own
# GPU builds of PyTorch and NumPy.
!pip install -q -e . 2>&1 | tail -n 1

## 5. Find the uploaded data

In [ ]:
from pathlib import Path

key = next(Path("/kaggle/input").rglob("answer_key.json"))
cv_dir = next(p.parent for p in Path("/kaggle/input").rglob("cv_a_data_engineer.docx"))
jd_dir = next(p.parent for p in Path("/kaggle/input").rglob("jds_1.txt"))
print(key, cv_dir, jd_dir, sep="\n")

## 6. Run the eval

All 12 pairs in one go (Kaggle has enough memory, so no batches are needed).
Progress is shown line by line and also saved to `eval.log`. If the session
stops, running this cell again continues from the last finished pair.

In [ ]:
import sys

command = [
    sys.executable, "-u", "scripts/run_eval.py",
    "--key", str(key), "--cv-dir", str(cv_dir), "--jd-dir", str(jd_dir),
    "--out", "/kaggle/working/kaggle_qwen7b.json", "--keep-loaded",
]  # fmt: skip
run_env = dict(os.environ, LLM_PROVIDER="ollama", PYTHONIOENCODING="utf-8")
with open("/kaggle/working/eval.log", "w") as eval_log:
    process = subprocess.Popen(
        command, env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in process.stdout:
        eval_log.write(line)
        eval_log.flush()
        if "warning" not in line.lower():
            print(line, end="", flush=True)
if process.wait() != 0:
    print("\n--- last lines of ollama.log ---")
    print("\n".join(Path("/kaggle/working/ollama.log").read_text().splitlines()[-20:]))
    raise RuntimeError(f"The eval failed (exit code {process.returncode}); see eval.log above")

## 7. Results

In [ ]:
summary = json.loads(Path("/kaggle/working/kaggle_qwen7b.json").read_text())["summary"]
fields = (
    "strict_accuracy", "lenient_accuracy", "false_met", "score_mean_abs_error",
    "negatives_passed", "extraction_coverage", "total_seconds",
)  # fmt: skip
print(json.dumps({k: summary[k] for k in fields}, indent=2))

Download `kaggle_qwen7b.json` from the *Output* tab and put it in
`data/eval/results/` on the laptop to compare with `round1_7b.json`.